# 🔋 EV Rental Dataset — Exploratory Data Analysis

This notebook presents a structured analysis of an electric vehicle (EV) rental dataset. The goal is to uncover patterns in battery usage, customer behaviour, and vehicle utilisation through a series of targeted experiments.

## 1. Setup

### 1.1 Importing Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression

### 1.2 Loading the Dataset

The dataset is loaded and inspected using `.head()` and `.info()` to understand its structure, data types, and initial shape.

In [ ]:
data=pd.read_csv(r'C:\Users\Surya\Documents\Free2Move.csv')
data.head()
data.info()

### 1.3 Feature Engineering

Two derived columns are created to support downstream analysis:

- **`used_time`** — rental duration, computed as the difference between `FINISHED_TIME` and `STARTED_TIME` (both converted from object to `datetime`).
- **`charge_used`** — battery consumed during the rental, computed as the difference between `CHARGELEVELSTART` and `CHARGELEVELEND`.

In [ ]:
data['STARTED_TIME'] = pd.to_datetime(data['STARTED_TIME'])
data['FINISHED_TIME'] = pd.to_datetime(data['FINISHED_TIME'])
data['used_time'] =data['FINISHED_TIME'] - data['STARTED_TIME']  
data["charge_used"] = data["CHARGELEVELSTART"] - data["CHARGELEVELEND"]

### 1.4 Outlier Identification

A separate DataFrame is created to isolate rentals with a duration exceeding 24 hours. These **133 outliers** are flagged for reference and excluded from core analysis to avoid skewing results.

In [ ]:
outliers=data[data['used_time'].dt.days > 0]

---

## 2. Data Quality Assessment

### 2.1 Missing Values — `CHARGELEVELSTART`

A boolean `missing` flag is created for rows where `CHARGELEVELSTART` is null. Grouping by this flag and computing column means reveals how missing values relate to other variables — particularly `SERVICERENTAL`.

In [ ]:
data['missing'] = data['CHARGELEVELSTART'].isna()
data.groupby('missing').mean(numeric_only=True)

> **Finding:** Null values in `CHARGELEVELSTART` have a pronounced impact on the `SERVICERENTAL` variable, suggesting that missing charge data is disproportionately associated with service-agent trips rather than customer rentals.

*Dropping the temporary `missing` flag column before proceeding.*

In [ ]:
data = data.drop(columns=['missing'])

### 2.2 Missing Values — `CHARGELEVELEND`

The same approach is applied to `CHARGELEVELEND` — creating a missing flag and comparing column means across the two groups.

In [ ]:
data['missing'] = data['CHARGELEVELEND'].isna()
data.groupby('missing').mean(numeric_only=True)

> **Finding:** Null values in `CHARGELEVELEND` show minimal impact on other variables and are not considered a significant concern for this analysis.

### 2.3 Splitting the Dataset

Based on the above findings, the dataset is split into two separate DataFrames:

- **`customer_data`** — rentals where `SERVICERENTAL == False`
- **`service_data`** — trips made by service agents where `SERVICERENTAL == True`

This separation is critical, as the two groups exhibit fundamentally different behaviour.

In [ ]:
customer_data = data[data['SERVICERENTAL'] == False]
service_data  = data[data['SERVICERENTAL'] == True]

A further subset, **`customer_data_no_charge`**, is created from `customer_data` by filtering for rentals where the customer did not charge the vehicle. This isolates organic usage patterns uninfluenced by charging stops.

In [ ]:
customer_data_no_charge = customer_data[customer_data["CHARGED"] == False].copy()
customer_data_no_charge

---

## 3. Analysis

### Experiment 1 — Rental Duration vs. Battery Consumption

**Hypothesis:** Longer rentals consume more battery charge.

The correlation coefficient between `used_time_h` (rental duration in hours) and `charge_used` is computed using `.corr()`. Null values and negative charge readings are dropped prior to calculation.

In [ ]:
customer_data_no_charge["used_time_h"] = (customer_data_no_charge["used_time"].dt.total_seconds() / 3600)
corr = customer_data_no_charge[["used_time_h", "charge_used"]].corr()
customer_data_no_charge = customer_data_no_charge.dropna()
customer_data_no_charge = customer_data_no_charge[customer_data_no_charge["charge_used"] >= 0]
corr                                           

> **Result:** The correlation between rental duration and battery consumption is very weak, **disproving the hypothesis**. Duration alone is not a reliable predictor of charge usage.

In [ ]:
plt.figure()
plt.scatter(
    customer_data_no_charge["used_time_h"],
    customer_data_no_charge["charge_used"],
    alpha=0.1
)
plt.xlabel("Used Time-hours")
plt.ylabel("Charge Used %")
plt.xlim(0, 10)
plt.title("Battery Used vs Rental Duration")
plt.show()

> **Observation:** The scatter plot (limited to rentals under 10 hours) confirms no clear trend. The majority of rentals last under 2 hours and consume less than 20% battery, with no discernible pattern linking time to charge depletion.

### Experiment 2 — Starting Charge Level vs. Battery Consumption

**Hypothesis:** Vehicles with a higher starting charge level will show lower percentage-wise battery consumption, as depletion rates may vary across the charge curve.

In [ ]:
plt.figure()
plt.scatter(
    customer_data_no_charge["CHARGELEVELSTART"],
    customer_data_no_charge["charge_used"],
    alpha=0.1
)
plt.xlabel("Starting_Charge")
plt.ylabel("Charge Used (%)")
plt.title("Battery Used vs Charge Start Level")
plt.show()

> **Observation:** The scatter plot reveals minimal rentals initiated below the 20% charge mark. Most rentals consume under 20% of battery regardless of starting level, suggesting starting charge does not significantly influence consumption rate.

*Null values and the temporary `missing` column are dropped from `customer_data` to ensure clean analysis going forward.*

In [ ]:
customer_data = customer_data.drop(columns=['missing'])
customer_data = customer_data.dropna()
customer_data.info()
customer_data.head()

### Experiment 3 — Rental Frequency by Charge Start Level

Customer rental records are bucketed into 10% charge-level intervals. For each bucket, the rental count and its share of total rentals (probability %) are computed.

In [ ]:
bins = range(0, 110, 10)
bucket = pd.cut(customer_data["CHARGELEVELSTART"], bins=bins)
cus_charge = bucket.value_counts().sort_index().reset_index()
cus_charge.columns = ["split", "count"]

cus_charge["probability %"] = cus_charge["count"] / cus_charge["count"].sum() * 100
cus_charge

> **Finding:** Rentals initiated below 20% charge are rare. Conversely, the 80–90% and 90–100% buckets account for the highest rental probabilities, indicating that customers strongly prefer well-charged vehicles.

The histogram below visualises the **distribution of rentals across starting charge levels**.

In [ ]:
plt.hist(customer_data["CHARGELEVELSTART"], bins)
plt.xlabel("Charge Level (%)")
plt.ylabel("Count")
plt.title("Distribution of Charge Start Level")
plt.show()

The bar chart below shows the **probability of a customer initiating a rental at each charge level bracket**.

In [ ]:
plt.bar(cus_charge["split"].astype(str), cus_charge["probability %"])
plt.xlabel("Charge Level Range (%)")
plt.ylabel("Probability")
plt.title("Probability of Customer Renting at Various Charge Levels")
plt.xticks(rotation=45)
plt.show()


### Experiment 4 — Vehicle Utilisation Frequency

The dataset contains **51 unique vehicles**. A utilisation table is constructed showing how often each vehicle ID appears in the rental records, alongside its probability of being selected.

In [ ]:
unique_models = data['VEHICLE_ID'].nunique()
unique_models
Model = data["VEHICLE_ID"].value_counts().reset_index().sort_index()
Model.columns = ["VEHICLE_ID", "count"]
Model["probability %"] = Model["count"] / Model["count"].sum() * 100
Model

> **Finding:** One vehicle (`dc753ee97d00780d5b36379e37ee78bf`) is rented significantly less frequently than all others, suggesting it may be experiencing operational issues or placement problems.

### Experiment 5 — End Charge Level Distribution: Customers vs. Service Agents

End charge levels are bucketed into 10% intervals separately for the customer and service datasets, allowing a direct comparison of how each group leaves vehicles.

In [ ]:
bucket = pd.cut(customer_data["CHARGELEVELEND"], bins=bins)
bucket2 = pd.cut(service_data["CHARGELEVELEND"], bins=bins)

CHARGELEVELEND_service = bucket2.value_counts().reset_index().sort_index()
CHARGELEVELEND_customer = bucket.value_counts().reset_index().sort_index()

CHARGELEVELEND_service.columns = ["CHARGELEVELEND", "count"]
CHARGELEVELEND_customer.columns = ["CHARGELEVELEND", "count"]
CHARGELEVELEND_customer = CHARGELEVELEND_customer.sort_values("CHARGELEVELEND").reset_index(drop=True)
CHARGELEVELEND_service = CHARGELEVELEND_service.sort_values("CHARGELEVELEND").reset_index(drop=True)

CHARGELEVELEND_service

> **Finding (Service Data):** Service agents are most active on vehicles with end charge below 20% or above 90%, consistent with a rebalancing or charging role.

> **Finding (Customer Data):** Customers rarely return vehicles at critically low charge levels, which reduces the recharging burden on service agents.

In [ ]:
CHARGELEVELEND_customer

### Experiment 6 — In-Rental Charging Behaviour

A count of customers who charged the vehicle during their rental (`CHARGED == True`) vs. those who did not is examined.

In [ ]:
customer_charging = customer_data["CHARGED"].value_counts().reset_index()
customer_charging.columns = ["CHARGED", "count"]

customer_charging

### Experiment 7 — Rental Duration vs. Charging Behaviour

Rentals are grouped into duration bins (0–1h, 1–2h, 2–4h, 4–8h, 8h+) and cross-tabulated with whether the vehicle was charged. The percentage of charging rentals in each bin is computed.

In [ ]:
bins_d = pd.to_timedelta([0, 1, 2, 4, 8, 24], unit="h")
labels = ["0–1", "1–2", "2–4", "4–8", "8+"]

customer_data["duration_bin"] = pd.cut(customer_data["used_time"],bins=bins_d,labels=labels,right=False)
charging_vs_duration = (customer_data.groupby(["duration_bin", "CHARGED"]).size().unstack(fill_value=0))
charging_vs_duration["percent_charged"] = (charging_vs_duration[True] /charging_vs_duration.sum(axis=1)) * 100
charging_vs_duration

> **Finding:** The likelihood of a customer charging the vehicle increases with rental duration. Customers on longer trips are more inclined to top up the battery.

### Experiment 8 — Starting Charge Level vs. Charging Behaviour

Similar to Experiment 7, but cross-tabulating starting charge level bins against charging behaviour to test whether low initial charge prompts customers to charge.

In [ ]:
customer_data["start_charge_bin"] = pd.cut(customer_data["CHARGELEVELSTART"],bins=bins)
chargelevel_vs_charged = (customer_data.groupby(["start_charge_bin", "CHARGED"]).size().unstack(fill_value=0))
chargelevel_vs_charged["percent_charged"] = (chargelevel_vs_charged[True] /chargelevel_vs_charged.sum(axis=1)) * 100

chargelevel_vs_charged

> **Finding:** Customers are significantly more likely to charge the vehicle when the starting charge level is below 30%, suggesting a comfort threshold that influences charging decisions.

### Experiment 9 — Duration × Starting Charge Matrix

A two-dimensional pivot matrix is constructed cross-tabulating starting charge bins against duration bins. This reveals compound preference patterns across both dimensions.

In [ ]:
chargelevel_vs_duration = (customer_data.groupby(["start_charge_bin", "duration_bin"]).size().unstack(fill_value=0))
chargelevel_vs_duration

> **Finding:** A diagonal preference boundary is visible in the matrix — customers favour higher starting charge even for short trips. A clear threshold emerges around the 30–40% charge range: below it, rentals drop off sharply across all duration bins.

### Experiment 10 — Per-Vehicle Charge Performance

For each vehicle, the average starting charge level is computed. A secondary metric identifies vehicles that are frequently rented below 20% charge, expressed as a proportion of that vehicle's total rentals.

In [ ]:
avg_charge_per_vehicle = (customer_data.groupby("VEHICLE_ID")["CHARGELEVELSTART"].mean().reset_index())
low_charge_vehicle = (customer_data[customer_data["CHARGELEVELSTART"] <20].groupby("VEHICLE_ID").size()
                      .reset_index(name="low_charge_rentals"))
low_charge_vehicle = low_charge_vehicle.sort_values("low_charge_rentals").reset_index(drop=True)
final_df = (low_charge_vehicle.merge(avg_charge_per_vehicle, on="VEHICLE_ID", how="left").merge(Model, on="VEHICLE_ID", how="left"))
final_df = final_df.sort_values("low_charge_rentals").reset_index(drop=True)
final_df["low_charge_probability"] = (final_df["low_charge_rentals"] / final_df["count"]) * 100
final_df


> **Finding:** Vehicle `dc753ee97d00780d5b36379e37ee78bf` consistently underperforms in rental frequency. Vehicle `1c347a2ffbbdfb70197160e8e77c9a49` is repeatedly rented at low charge levels — while the proportion is not yet alarming, sustained low-charge cycling may accelerate battery degradation over time.

---

## 4. Summary

The ten experiments above highlight several key behavioural patterns in EV rental usage:

| # | Experiment | Key Finding |
|---|-----------|-------------|
| 1 | Duration vs. Battery Used | Weak correlation — duration does not predict charge consumption |
| 2 | Start Charge vs. Battery Used | No meaningful relationship; most trips use under 20% battery |
| 3 | Rental Frequency by Charge Level | Strong preference for vehicles above 80% charge |
| 4 | Vehicle Utilisation | One vehicle is significantly underutilised |
| 5 | End Charge — Customer vs. Agent | Agents handle low-charge recovery; customers return vehicles reasonably charged |
| 6 | In-Rental Charging | Most customers do not charge during rentals |
| 7 | Duration vs. Charging | Longer rentals correlate with higher charging rates |
| 8 | Start Charge vs. Charging | Customers more likely to charge when starting below 30% |
| 9 | Duration × Start Charge Matrix | Diagonal preference boundary at ~30–40% starting charge |
| 10 | Per-Vehicle Performance | Two vehicles flagged for anomalous usage patterns |